# Lecture 3.8 — Error Handling Inside Tools

Tools fail in the real world. APIs go down, databases time out, and models occasionally send malformed arguments. This notebook walks through the full error handling surface the OpenAI Agents SDK gives you for function tools: the default behaviour, a custom `failure_error_function`, re-raising with `None`, timeout configuration, and the full set of SDK exceptions you may need to catch.

## Cell 1: Install the SDK

This notebook uses the `openai-agents` package. The install below pins a specific version so the examples in this notebook behave consistently regardless of when you run it. If the package is already present in this Colab session (for example, you ran another notebook earlier today), this cell finishes almost instantly since pip detects it's already satisfied.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.19.4 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.5/968.5 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00


## Cell 2: Configure Your API Key

We use Google Colab's Secrets manager to store the API key, so it's never hardcoded in the notebook itself.

**Steps to add your key in Colab:**
1. Click the key icon (🔑) in the left sidebar to open the Secrets panel.
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY` and paste your key as the value.
4. Toggle **Notebook access** on for this notebook.

**Running locally instead of Colab?** Skip the `userdata` call below and set the key as an environment variable in your terminal before launching Jupyter, for example: `export OPENAI_API_KEY="sk-..."`.

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3: Set the Model Name

We declare a single `MODEL_NAME` variable here and reuse it in every `Agent` definition in this notebook. Changing this one line updates the model used throughout the entire notebook, so you don't need to hunt through every cell if you want to try a different model.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4: Imports

| Import | Purpose |
|---|---|
| `asyncio` | Used to simulate slow operations with `asyncio.sleep()` and to run async error handlers |
| `Any` (typing) | Type hint for `RunContextWrapper[Any]` in error handler signatures |
| `Reasoning` | Controls reasoning effort in `ModelSettings`. Imported from `openai.types.shared`, **not** from `agents` |
| `Agent`, `Runner`, `function_tool`, `ModelSettings` | Core SDK building blocks used throughout the course |
| `RunContextWrapper` | The context object passed as the first argument to error handler functions |
| `ModelBehaviorError`, `ToolTimeoutError`, `UserError` | SDK exceptions this lecture covers. All three import from the `agents` top-level package |
| `default_tool_error_function` | The built-in error formatter used when you don't pass `failure_error_function`. This one is the exception: it imports from `agents.tool`, not from `agents` top-level |

Keep that last point in mind. It's a common mix-up: most SDK exceptions and classes come from `agents`, but `default_tool_error_function` lives in `agents.tool`.

In [4]:
import asyncio
from typing import Any

from openai.types.shared import Reasoning

from agents import (
    Agent,
    ModelSettings,
    ModelBehaviorError,
    RunContextWrapper,
    Runner,
    ToolTimeoutError,
    UserError,
    function_tool,
)
from agents.tool import default_tool_error_function

## Cell 5: The Three Error Handling Modes

Every function tool created with `@function_tool` accepts a `failure_error_function` parameter. It controls what happens when the tool's Python code raises an exception, or when the model sends arguments that fail to parse.

**Mode 1 — Default (omit `failure_error_function` entirely)**
The SDK runs `default_tool_error_function` automatically. It catches the exception, formats it into a string, and sends that string back to the model as the tool's result. For a general exception it returns something like *"An error occurred while running the tool. Please try again. Error: ..."*. For a JSON parsing failure (invalid arguments from the model) it returns a more specific message about retrying with valid JSON. Either way, the run keeps going. The model sees the failure and can decide what to do next.

**Mode 2 — Custom function (`failure_error_function=my_fn`)**
You supply your own formatter with the signature `(ctx: RunContextWrapper[Any], error: Exception) -> str`. It can be a regular function or an `async def` function; the SDK awaits it automatically if it's awaitable. Inside your handler, `ctx.context` gives you access to whatever application state you passed into the run, so you can log to a database, fire an alert, or personalise the message. Whatever string you return is what the model sees. The run continues.

**Mode 3 — `failure_error_function=None`**
Errors are not swallowed at all. They propagate out of `Runner.run()` for you to catch. This could be a `ModelBehaviorError` if the model produced invalid JSON, or whatever exception your tool code raised. Use this for tools where continuing after a silent failure would be unsafe.

## Cell 6: Mode 1 — Default Error Handling

This is the simplest case: a tool that can fail, with no `failure_error_function` specified. `flaky_api_call` raises a `ValueError` for any user ID other than `"user_123"`. Because we haven't passed `failure_error_function`, the SDK automatically wraps that exception using `default_tool_error_function` before sending anything to the model. Watch how the agent's `instructions` tell it what to do when a tool fails: apologise and suggest trying again. That's the model reacting to the formatted error string, not to a crash.

In [5]:
@function_tool
def flaky_api_call(user_id: str) -> str:
    """Fetches a user profile from an API.

    Args:
        user_id: The user ID to look up.
    """
    if user_id == "user_123":
        return "Profile: Alice, Premium member since 2022."
    raise ValueError(
        f"API error: user_id '{user_id}' not found in system."
    )


agent = Agent(
    name="Profile Agent",
    instructions=(
        "You are a helpful assistant. "
        "Use the flaky_api_call tool to look up user profiles."
        " If a tool fails, apologise and suggest trying again."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[flaky_api_call],
)

result_ok = await Runner.run(agent, "Look up user_123.")
print("Success:", result_ok.final_output)

result_fail = await Runner.run(agent, "Look up user_999.")
print("Failure (default):", result_fail.final_output)

Success: Alice — Premium member since 2022.
Failure (default): Sorry, I couldn’t find user_999. Please try again with a different user ID.


## Cell 7: Inspect the Default Error Function Directly

`default_tool_error_function` is just a regular Python function with the signature `(ctx, error) -> str`, the exact same shape as any custom handler you'll write in Mode 2. Calling it directly here, outside of a real tool run, shows you exactly what string the model receives when a tool fails with the default configuration. Seeing this output makes it much clearer what you're replacing when you write a custom handler in the next cell.

In [6]:
class _FakeCtx:
    context = None


test_error = ValueError("Something went wrong. This user doesnt exist!")
error_msg = default_tool_error_function(_FakeCtx(), test_error)

print("Default error message sent to model:")
print(error_msg)

Default error message sent to model:
An error occurred while running the tool. Please try again. Error: Something went wrong. This user doesnt exist!


## Cell 8: Mode 2 — Custom `failure_error_function` (Sync)

Here we pass our own formatter to `@function_tool(failure_error_function=my_custom_error_function)`. Inside `my_custom_error_function`, we log the raw exception to the console (in a real app this could be a call to your logging or alerting system) and return a friendlier, user-facing string. Notice the signature matches `default_tool_error_function` exactly: `(context, error) -> str`. The `context` parameter is a `RunContextWrapper[Any]`, and `context.context` would give you access to any app-level state you passed into `Runner.run()`, which is useful for including user-specific detail in the log or the message.

In [8]:
def my_custom_error_function(
    context: RunContextWrapper[Any],
    error: Exception,
) -> str:
    print(f"[LOG] Tool error: {type(error).__name__}: {error}")
    return (
        "I encountered an issue retrieving that information. "
        "The system returned: service temporarily unavailable."
        " Please ask the user to try again in a moment."
    )


@function_tool(failure_error_function=my_custom_error_function)
def get_user_profile(user_id: str) -> str:
    """Fetches a user profile.

    Args:
        user_id: The user ID to look up.
    """
    if user_id == "user_123":
        return "Profile: Alice, Premium member since 2022."
    raise ValueError(f"User '{user_id}' not found.")


custom_agent = Agent(
    name="Custom Error Agent",
    instructions=(
        "You are a helpful assistant. "
        "Use the get_user_profile tool to look up users."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[get_user_profile],
)

result = await Runner.run(custom_agent, "Look up user_999.")
print("Custom error response:", result.final_output)

ValidationError: 1 validation error for InputTokensDetails
cache_write_tokens
  Field required [type=missing, input_value={'cached_tokens': 0}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

## Cell 9: Mode 2 — Async Custom Error Function

`failure_error_function` doesn't have to be a plain sync function. Here `async_error_handler` is an `async def`, and the SDK detects that its return value is awaitable and awaits it automatically before sending the result to the model. This matters when your error handling itself needs to do async work, such as writing to a database or calling an external logging service, without blocking the rest of the run.

In [ ]:
async def async_error_handler(
    context: RunContextWrapper[Any],
    error: Exception,
) -> str:
    await asyncio.sleep(0.01)  # simulate external logging
    print(f"[ASYNC LOG] Tool failed: {error}")
    return (
        f"The tool encountered an error: {type(error).__name__}."
        " Our team has been notified. Please try again."
    )


@function_tool(failure_error_function=async_error_handler)
async def async_flaky_tool(query: str) -> str:
    """An async tool that sometimes fails.

    Args:
        query: The query to process.
    """
    if query == "fail":
        raise RuntimeError("Simulated async tool failure.")
    return f"Result for query: {query}"


async_agent = Agent(
    name="Async Error Agent",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[async_flaky_tool],
)

result = await Runner.run(async_agent, "Query: fail")
print("Async error response:", result.final_output)

[ASYNC LOG] Tool failed: Simulated async tool failure.
Async error response: The tool failed with a RuntimeError. Please try again.


## Cell 10: Mode 3 — `failure_error_function=None` (Re-raise)

Passing `None` explicitly turns off the SDK's error swallowing for this tool. Any exception raised inside the tool, or any `ModelBehaviorError` from invalid model-generated JSON, propagates straight out of `Runner.run()`. This is the right choice for tools where silently telling the model "try again" would be worse than stopping the run entirely, for example a tool that submits a payment or writes to a critical system. Because the exception is no longer caught for you, you need a `try`/`except` around the `Runner.run()` call yourself, as shown below.

**Why this tool fails unconditionally:** an earlier version of this example asked the model to call a tool with an obviously invalid argument (a negative number), expecting the tool's own validation to raise. That turned out to be unreliable. The model would often substitute a valid value on its own rather than pass through the one it was told to use, since nothing forces it to honor a specific argument value even with `tool_choice="required"`. So instead, `submit_critical_payment` fails every time it's called, regardless of what argument the model supplies, simulating a payment gateway that is genuinely down. This makes the demo deterministic: as long as the tool is called at all, the `RuntimeError` fires and propagates out of `Runner.run()` for us to catch.

In [ ]:
@function_tool(failure_error_function=None)
def submit_critical_payment(amount: float) -> str:
    """Submits a payment through a critical, single-attempt payment gateway.

    Args:
        amount: The payment amount in dollars.
    """
    raise RuntimeError(
        "Payment gateway is currently unreachable. "
        "This operation cannot be retried automatically."
    )


strict_agent = Agent(
    name="Payments Agent",
    instructions=(
        "You are a payments assistant. "
        "Use submit_critical_payment to process the user's payment request."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[submit_critical_payment],
)

try:
    result = await Runner.run(
        strict_agent,
        "Submit a payment of $250.",
    )
    print("Result:", result.final_output)
except Exception as e:
    print(f"Run failed: {type(e).__name__}: {e}")


Run failed: UserError: Error running tool submit_critical_payment: Payment gateway is currently unreachable. This operation cannot be retried automatically.


## Cell 11: Timeout — `error_as_result` (Default Behaviour)

`@function_tool` also accepts a `timeout` in seconds. If the tool doesn't finish within that window, the SDK steps in. With `timeout_behavior="error_as_result"` (the default when you set a `timeout`), the SDK sends a model-visible timeout message back instead of letting the tool hang, and the run continues. The default message looks like `"Tool 'slow_database_query' timed out after 2 seconds."`.

One important constraint: timeout only works for **async** `@function_tool` handlers. Sync tools run inside a background thread, and Python gives the SDK no way to interrupt a thread mid-execution, so a `timeout` on a sync tool has no effect. `slow_database_query` below is `async def` specifically so the timeout can actually apply.

In [ ]:
@function_tool(
    timeout=2.0,
    timeout_behavior="error_as_result",
)
async def slow_database_query(table_name: str) -> str:
    """Queries a database table. May be slow.

    Args:
        table_name: The table to query.
    """
    await asyncio.sleep(5.0)
    return f"Data from {table_name}."


timeout_agent = Agent(
    name="Timeout Agent",
    instructions=(
        "You are a database assistant. "
        "Use slow_database_query to fetch data. "
        "If a query times out, inform the user."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[slow_database_query],
)

result = await Runner.run(
    timeout_agent,
    "Query the users table.",
)
print("Timeout (error_as_result):", result.final_output)

Timeout (error_as_result): The query to the `users` table timed out. Please try again later.


## Cell 12: Timeout — `raise_exception`

Set `timeout_behavior="raise_exception"` when a timeout should stop the run rather than get handed back to the model. Instead of a friendly message, the SDK raises `ToolTimeoutError`, which propagates out of `Runner.run()` just like the `None` case in Cell 10. `ToolTimeoutError` carries two useful fields: `tool_name`, the name of the tool that timed out, and `timeout_seconds`, the configured limit. You catch it at the `Runner.run()` call site, exactly like any other exception.

In [ ]:
@function_tool(
    timeout=1.5,
    timeout_behavior="raise_exception",
)
async def hard_timeout_tool(query: str) -> str:
    """A tool with hard timeout — failure stops the run.

    Args:
        query: The query to process.
    """
    await asyncio.sleep(5.0)
    return "Done."


hard_agent = Agent(
    name="Hard Timeout Agent",
    instructions="You are a helpful assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[hard_timeout_tool],
)

try:
    result = await Runner.run(
        hard_agent,
        "Run the tool.",
    )
    print("Result:", result.final_output)
except ToolTimeoutError as e:
    print(
        f"Tool timed out: {e.tool_name} "
        f"after {e.timeout_seconds}s"
    )

Tool timed out: hard_timeout_tool after 1.5s


## Cell 13: Custom `timeout_error_function`

When you're using `timeout_behavior="error_as_result"` but want to replace the generic default timeout message with something more specific, pass `timeout_error_function`. It takes the exact same `(ctx, error) -> str` signature you saw with `failure_error_function` back in Cell 8. This is useful when you want the message the model sees to hint at *why* a query might be slow, such as high load, rather than just repeating the raw timeout duration.

In [ ]:
def custom_timeout_message(
    context: RunContextWrapper[Any],
    error: Exception,
) -> str:
    return (
        "The database query took too long to respond. "
        "This usually means high load. "
        "Please retry in 30 seconds."
    )


@function_tool(
    timeout=2.0,
    timeout_behavior="error_as_result",
    timeout_error_function=custom_timeout_message,
)
async def custom_timeout_tool(table_name: str) -> str:
    """Queries a table with a custom timeout message.

    Args:
        table_name: The table to query.
    """
    await asyncio.sleep(5.0)
    return f"Data from {table_name}."


ct_agent = Agent(
    name="Custom Timeout Agent",
    instructions=(
        "You are a database assistant. "
        "Use custom_timeout_tool to query data."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[custom_timeout_tool],
)

result = await Runner.run(
    ct_agent,
    "Query the orders table.",
)
print("Custom timeout response:", result.final_output)

Custom timeout response: The `orders` query timed out due to high load.

Please retry in about 30 seconds.


## Cell 14: Full Exceptions Reference

Everything in this lecture has revolved around a handful of SDK exception classes. Here's the complete reference, all importable from `agents` top-level:

| Exception | When it's raised | Import |
|---|---|---|
| `AgentsException` | Base class every other SDK exception inherits from | `from agents import AgentsException` |
| `MaxTurnsExceeded` | A run exceeds the configured `max_turns` limit | `from agents import MaxTurnsExceeded` |
| `ModelBehaviorError` | The model produced invalid or unparseable output, such as malformed JSON tool arguments | `from agents import ModelBehaviorError` |
| `ToolTimeoutError` | A tool exceeds its `timeout` with `timeout_behavior="raise_exception"` | `from agents import ToolTimeoutError` |
| `UserError` | The SDK was misconfigured or misused, a mistake in your own code rather than the model's | `from agents import UserError` |
| `InputGuardrailTripwireTriggered` | An input guardrail's tripwire fires before the run proceeds | `from agents import InputGuardrailTripwireTriggered` |
| `OutputGuardrailTripwireTriggered` | An output guardrail's tripwire fires on the agent's response | `from agents import OutputGuardrailTripwireTriggered` |

Guardrails are covered in their own lecture later in the course. They're included here so you have the complete exception picture in one place.

## Cell 15: Error Handling Decision Guide

A quick reference for choosing the right configuration when you build your own tools:

| Scenario | Configuration |
|---|---|
| Recoverable failure, the model can retry or adapt | Default (omit `failure_error_function`) |
| Need a user-friendly message plus logging or alerting | Custom `failure_error_function` |
| Critical operation that must stop the run on failure | `failure_error_function=None` |
| Slow tool where the model should be told and can retry | `timeout=N`, `timeout_behavior="error_as_result"` |
| Slow tool where a timeout should be treated as fatal | `timeout=N`, `timeout_behavior="raise_exception"` |
| Need a custom message when a timeout is reported to the model | `timeout_error_function=my_fn` |

With this, you now have the complete error handling toolkit for function tools: three failure modes, two timeout behaviours, a way to customise either message, and the full exceptions reference to catch anything that propagates.